In [1]:
import json
json_path = "file_dicts.json"
with open(json_path, "r") as f:
    file_dicts = json.load(f)
file_dicts[0]
import nibabel as nib 
for v in file_dicts[0].values():
    if v.endswith(".nii.gz"):
        img = nib.load(v).get_fdata()
        print(img.shape)
file_dict_one = []
for file_dict in file_dicts:
    file_dict_one.append({"image":file_dict["image"]})
    file_dict_one.append({"image":file_dict["source_image"]})
print(len(file_dict_one))

(256, 256, 150, 1)
(256, 256, 150, 1)
(256, 256, 150, 1)
20656


In [2]:
import monai 
from monai.bundle.config_parser import ConfigParser
import torch 
# !mkdir models
# # !cp  /data1/syliu/miccai24_maisi_SR/models/autoencoder_epoch273.pt models/autoencoder_epoch273.pt 
# config= ConfigParser()
# config.read_config("shiyu_utils/config_maisi3d-rflow.json")
# autoencoder = config.get_parsed_content("autoencoder_def")
# autoencoder.load_state_dict(torch.load("models/autoencoder_epoch273.pt"))

In [3]:
from shiyu_utils.maisi_transforms import VAE_Transform
transform = VAE_Transform(
    is_train=False,
    random_aug=False,
    val_patch_size=(64,84,84),
    output_dtype=torch.float32,
    spacing_type="fixed",
    spacing=(3.,3.,3.),
    image_keys=["image"]
)
transform = transform.transform_dict["ct"]
transform.transforms
transform.transforms[-4].scaler.a_min=-10
transform.transforms[-4].scaler.a_max=0
transform

In [4]:
import random 
for _ in range(10):
    index= random.randint(0,len(file_dicts)-1)
    data = transform(file_dicts[index])
    for v in data.values():print(v.shape)

torch.Size([1, 64, 84, 84])
torch.Size([1, 64, 84, 84])
torch.Size([1, 64, 84, 84])
torch.Size([1, 64, 84, 84])
torch.Size([1, 64, 84, 84])
torch.Size([1, 64, 84, 84])
torch.Size([1, 64, 84, 84])
torch.Size([1, 64, 84, 84])
torch.Size([1, 64, 84, 84])
torch.Size([1, 64, 84, 84])


In [5]:
import monai.data as md 
dataset = md.Dataset(data=file_dict_one,transform=transform)
dataloader = md.DataLoader(dataset,batch_size=1,shuffle=False,num_workers=32)
save_data_root ="ixi_mcx_2025_lowres"
import os 
import torch 
import shutil 
os.makedirs(save_data_root,exist_ok=True)
#autoencoder = autoencoder.cuda().eval()
from tqdm import  tqdm
for batch in tqdm(dataloader):
    file_name=batch["image"].meta["filename_or_obj"][0].replace(".nii","").replace(".gz","")
    base_name = os.path.basename(file_name)
    label_name = os.path.basename(os.path.dirname(file_name))
    subject_name = os.path.basename(os.path.dirname(os.path.dirname(os.path.dirname(file_name))))
    save_path = os.path.join(save_data_root,subject_name+"_"+base_name+"_"+label_name+".pt")
    # if os.path.exists(save_path):
    #     continue
    with torch.no_grad(),torch.cuda.amp.autocast(True):
        #latent = autoencoder.encode_stage_2_inputs(batch['image'].float().cuda())
        latent = batch["image"].float().cuda()
        latent = latent.cpu().detach()
        torch.save(latent,save_path.replace(".nii.gz",".pt"))
        print("saving latent:",save_path)

  0%|          | 0/20656 [00:00<?, ?it/s]/tmp/ipykernel_3873015/3476653511.py:19: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(),torch.cuda.amp.autocast(True):
  0%|          | 7/20656 [00:04<2:58:25,  1.93it/s] 

saving latent: ixi_mcx_2025_lowres/m2m_IXI012-HH-_MCX_FT7_p250_t8_results_log_full.pt
saving latent: ixi_mcx_2025_lowres/m2m_IXI012-HH-_MCX_FT7_p250_t8_results_log_simple.pt
saving latent: ixi_mcx_2025_lowres/m2m_IXI012-HH-_MCX_AF3_p250_t8_results_log_full.pt
saving latent: ixi_mcx_2025_lowres/m2m_IXI012-HH-_MCX_AF3_p250_t8_results_log_simple.pt
saving latent: ixi_mcx_2025_lowres/m2m_IXI012-HH-_MCX_FT10_p250_t8_results_log_full.pt
saving latent: ixi_mcx_2025_lowres/m2m_IXI012-HH-_MCX_FT10_p250_t8_results_log_simple.pt
saving latent: ixi_mcx_2025_lowres/m2m_IXI012-HH-_MCX_AF4_p250_t8_results_log_full.pt
saving latent: ixi_mcx_2025_lowres/m2m_IXI012-HH-_MCX_AF4_p250_t8_results_log_simple.pt
saving latent: ixi_mcx_2025_lowres/m2m_IXI012-HH-_MCX_FT8_p250_t8_results_log_full.pt
saving latent: ixi_mcx_2025_lowres/m2m_IXI012-HH-_MCX_FT8_p250_t8_results_log_simple.pt
saving latent: ixi_mcx_2025_lowres/m2m_IXI012-HH-_MCX_AF7_p250_t8_results_log_full.pt


  0%|          | 16/20656 [00:05<1:00:43,  5.67it/s]

saving latent: ixi_mcx_2025_lowres/m2m_IXI012-HH-_MCX_AF7_p250_t8_results_log_simple.pt
saving latent: ixi_mcx_2025_lowres/m2m_IXI012-HH-_MCX_FT9_p250_t8_results_log_full.pt
saving latent: ixi_mcx_2025_lowres/m2m_IXI012-HH-_MCX_FT9_p250_t8_results_log_simple.pt
saving latent: ixi_mcx_2025_lowres/m2m_IXI012-HH-_MCX_AF8_p250_t8_results_log_full.pt
saving latent: ixi_mcx_2025_lowres/m2m_IXI012-HH-_MCX_AF8_p250_t8_results_log_simple.pt
saving latent: ixi_mcx_2025_lowres/m2m_IXI012-HH-_MCX_Fp1_p250_t8_results_log_full.pt
saving latent: ixi_mcx_2025_lowres/m2m_IXI012-HH-_MCX_Fp1_p250_t8_results_log_simple.pt


  0%|          | 24/20656 [00:05<33:33, 10.24it/s]  

...

100%|█████████▉| 20625/20656 [14:03<00:00, 32.19it/s]

saving latent: ixi_mcx_2025_lowres/m2m_IXI653-Guy_MCX_FC2_p250_t8_results_log_full.pt
saving latent: ixi_mcx_2025_lowres/m2m_IXI653-Guy_MCX_FC2_p250_t8_results_log_simple.pt
saving latent: ixi_mcx_2025_lowres/m2m_IXI653-Guy_MCX_T7_p250_t8_results_log_full.pt
saving latent: ixi_mcx_2025_lowres/m2m_IXI653-Guy_MCX_T7_p250_t8_results_log_simple.pt
saving latent: ixi_mcx_2025_lowres/m2m_IXI653-Guy_MCX_FC3_p250_t8_results_log_full.pt
saving latent: ixi_mcx_2025_lowres/m2m_IXI653-Guy_MCX_FC3_p250_t8_results_log_simple.pt
saving latent: ixi_mcx_2025_lowres/m2m_IXI653-Guy_MCX_T8_p250_t8_results_log_full.pt
saving latent: ixi_mcx_2025_lowres/m2m_IXI653-Guy_MCX_T8_p250_t8_results_log_simple.pt
saving latent: ixi_mcx_2025_lowres/m2m_IXI653-Guy_MCX_FC4_p250_t8_results_log_full.pt
saving latent: ixi_mcx_2025_lowres/m2m_IXI653-Guy_MCX_FC4_p250_t8_results_log_simple.pt


100%|█████████▉| 20647/20656 [14:04<00:00, 46.07it/s]

saving latent: ixi_mcx_2025_lowres/m2m_IXI653-Guy_MCX_T9_p250_t8_results_log_full.pt
saving latent: ixi_mcx_2025_lowres/m2m_IXI653-Guy_MCX_T9_p250_t8_results_log_simple.pt
saving latent: ixi_mcx_2025_lowres/m2m_IXI653-Guy_MCX_FC5_p250_t8_results_log_full.pt
saving latent: ixi_mcx_2025_lowres/m2m_IXI653-Guy_MCX_FC5_p250_t8_results_log_simple.pt
saving latent: ixi_mcx_2025_lowres/m2m_IXI653-Guy_MCX_TP10_p250_t8_results_log_full.pt
saving latent: ixi_mcx_2025_lowres/m2m_IXI653-Guy_MCX_TP10_p250_t8_results_log_simple.pt
saving latent: ixi_mcx_2025_lowres/m2m_IXI653-Guy_MCX_FC6_p250_t8_results_log_full.pt
saving latent: ixi_mcx_2025_lowres/m2m_IXI653-Guy_MCX_FC6_p250_t8_results_log_simple.pt
saving latent: ixi_mcx_2025_lowres/m2m_IXI653-Guy_MCX_TP7_p250_t8_results_log_full.pt
saving latent: ixi_mcx_2025_lowres/m2m_IXI653-Guy_MCX_TP7_p250_t8_results_log_simple.pt
saving latent: ixi_mcx_2025_lowres/m2m_IXI653-Guy_MCX_FCz_p250_t8_results_log_full.pt
saving latent: ixi_mcx_2025_lowres/m2m_IXI65

100%|██████████| 20656/20656 [14:04<00:00, 24.46it/s]
